# Camada Silver

A camada silver recebe os dados da camada bronze e faz o tratamento dos dados, normalizando-os e formantando-os, de forma que os dados fiquem organizados e limpos para a próxima camada.

## 1. Definindo as variáveis do ambiente

Guardando os caminhos e os nomes das camadas nas variáveis para utilizar nas operações da camada:

In [0]:
# Obs: essa "inicialização" foi definida na camada bronze
catalog = "cine_data_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

#nome das camadas contacatenado com o nome do catálogo
bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## 1.1. Definindo as funções de checagem

Nesse bloco serão definidas funções que serão utilizadas para checar se os resultados da camada silver estão seguindo um padrão esperado.

In [0]:
# OBS: funções foram retiradas do notebook da aula
from pyspark.sql import functions as F, Row
from datetime import datetime

dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

# agrupa pela chave e conta quantas chaves aparecem mais de uma vez
def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

## 2. Organizando as tabelas
A partir daqui, vamos montar cada tabela da camada silver respectiva às tabelas da camada bronze, passando o tratamento adequado pra cada contexto.

### 2.1. `movies_info` -> `info_filmes`

A tabela das informações dos filmes que estão na camada bronze, agora vão ser normalizadas, formatadas, tratadas e adicionadas na tabela respectiva na camada silver.

### 2.1.1 Dados repetidos
Para fazer a parte da deduplicação (filtrar os dados repetidos), os dados serão organizados em uma "ficha" de prioridade, onde essa prioridade será baseada na data da ingestão daquele dado. Cada dado será enumerado (como se fosse um ranking). O primeiro fica e os últimos são excluidos. Isso vai ser baseado na coluna `id_filme`.

In [0]:
# guardando o local da camada bronze
df_bronze_movies_info = spark.table(f"{bronze_schema}.tb_movies_info")

#import do recurso window
from pyspark.sql import Window
from pyspark.sql.functions import countDistinct

#removendo os dados duplicados pelo id:

# tratando id_filme, descartando ids nulos e criando um window
window = Window.partitionBy(F.trim("id_filme")).orderBy(F.col("data_ingestao").desc())

#OBS: dropDuplicates não se encaixa bem nesse contexto porque ele não garante que o dado que vai ficar na tabela será o mais atualizado, por isso optei por fazer uma deduplicação manual com uma função window.

df_silver_info_filmes = (
    df_bronze_movies_info
        # renomeando as colunas existentes para as colunas da camada silver em portugues
        .select(F.col("id").alias("id_filme"), F.col("title").alias("titulo"), F.col("original_title").alias("titulo_original"), F.col("release_date").alias("data_lancamento"), F.col("runtime").alias("duracao_minutos"), F.col("original_language").alias("idioma_original"), F.col("status").alias("status_filme"), F.col("overview").alias("sinopse"), F.col("tagline").alias("frase_divulgacao"), F.col("ingestion_datetime").alias("data_ingestao"))
        #obs: ingestion_datetime permanece no momento para ser utilizada na hora da deduplicação, onde a função vai procurar o dado mais atualizado da tabela
        
        #criando a coluna temporaria ordem para fazer a deduplicação
        .withColumn("ordem", F.row_number().over(window))
        # filtrando quem está no topo da lista (deduplicacao) e removendo a coluna ordem
        .filter(F.col("ordem") == 1).drop("ordem")
        # removendo a coluna da data pois ela nao sera mais necessária
        .drop("data_ingestao")
)

#### 2.1.2 Normalização do status do filme
Os dados da coluna do status do filme vão ser extraídos e padronizados. Os dados não validos ou duplicados vao virar `null`:

In [0]:
# normalizando stauts do filme

df_silver_info_filmes =(
    df_silver_info_filmes
    # converter tudo para letras minusculas, para não ter problemas nas comparações
        .withColumn("status_filme", F.lower(F.trim(F.col("status_filme"))))
        #troca hifens e underlines por espaço
        .withColumn("status_filme", F.regexp_replace(F.col("status_filme"), "[-_]+", " "))
        #remove tudo o que não for letra minuscula ou espaço (pontuacao)
        .withColumn("status_filme", F.regexp_replace(F.col("status_filme"), "[^a-z ]", ""))
        #reduz espaços repetidos a um só
        .withColumn("status_filme", F.regexp_replace(F.col("status_filme"), r"\s+", " "))
        #tira os espaços do inicio e do fim
        .withColumn("status_filme", F.trim(F.col("status_filme")))

        #traduzindo status em ingles para portugues
        .withColumn("status_filme", F.when(F.col("status_filme") == "released", "Lançado")
                    .when(F.col("status_filme") == "post production", "Pós Produção")
                    .when(F.col("status_filme") == "planned", "Planejado")
                    .when(F.col("status_filme") == "in production", "Em Produção")
                    .when(F.col("status_filme") == "rumored", "Rumores")
                    .when(F.col("status_filme") == "canceled", "Cancelado")
                    .when(F.col("status_filme") == "cancelled", "Cancelado")
                    .otherwise("Não Informado"))
)

display(df_silver_info_filmes.groupBy("status_filme").count())

#### 2.1.3. Formatação da data de lançamento

Extraíndo as datas dos lançamentos dos filmes e convertendo para o formato dd-MM-yyyy.

In [0]:
# formatos existentes nos arquivos de entrada
formatos = ["yyyy-MM-dd", "MM-dd-yyyy", "dd-MM-yyyy", "MM/dd/yyyy"]  

# tenta converter para timestamp usando uma lista de formatos
tentativas = [
    F.try_to_timestamp(F.trim(F.col("data_lancamento")), F.lit(f)).cast("date")
    for f in formatos
]

# cria uma coluna com o resultado da primeira conversão que não falhou
df_silver_info_filmes = df_silver_info_filmes.withColumn("data_convertida", F.coalesce(*tentativas))

# adiciona uma coluna com o ano de lançamento
df_silver_info_filmes.select(F.min("data_convertida"), F.max("data_convertida")).show()

df_silver_info_filmes = (
    df_silver_info_filmes.drop("data_lancamento")
      .withColumnRenamed("data_convertida", "data_lancamento")
      .withColumn("ano_lancamento", F.year("data_lancamento"))
)

#### 2.1.4. Tratamento do campo da duração em minutos
Transformando o tempo em minutos que chegou em `string` e convertendo para inteiro:

In [0]:
#duracao minutos
# se for 0, vai ser 0 minutos
display(df_silver_info_filmes.groupBy("duracao_minutos").count())

#convertentdo duracao em minutos para inteiro
df_silver_info_filmes = df_silver_info_filmes.withColumn("duracao_minutos", F.expr("try_cast(trim(duracao_minutos) AS INT)"))

df_silver_info_filmes = df_silver_info_filmes.select(
    "id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao"
)

#### 2.1.5. Gravando na tabela

Com os dados organizados, eles serão gravados na tabela das informações do filme na camada silver.

In [0]:
#gravando na tabela

(df_silver_info_filmes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_info_filmes"))

# testando as funções de check
df_check = spark.table(f"{silver_schema}.tb_info_filmes")
tabela = "silver.tb_info_filmes"

status_validos = ["Lançado", "Pós Produção", "Em Produção", "Planejado",
                  "Rumores", "Cancelado", "Não Informado"]

dq_check(tabela, "id_filme não nulo", df_check, F.col("id_filme").isNotNull())
dq_check_unique(tabela, "id_filme único", df_check, ["id_filme"])
dq_check(tabela, "status_filme em domínio válido", df_check, F.col("status_filme").isin(status_validos))
dq_check(tabela, "duracao_minutos >= 0", df_check, F.col("duracao_minutos") >= 0)
dq_check(tabela, "ano_lancamento plausível", df_check, F.col("ano_lancamento").between(1880, 2100))

## 2.2. `movies_metrics` -> `metricas_engajamento`

In [0]:
# guardando o local da camada bronze
df_bronze_movies_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

df_silver_metricas_engajamento = (
    df_bronze_movies_metrics
        # renomeando as colunas existentes para as colunas da camada silver em portugues
        .select(F.col("id").alias("id_filme"), F.col("popularity").alias("popularidade"), F.col("vote_average").alias("nota_media_tmdb"), F.col("vote_count").alias("qtd_votos_tmdb"), F.col("averageRating").alias("nota_media_imdb"), F.col("numVotes").alias("qtd_votos_imdb"))
        
)

padrao_numero = r"^-?[0-9]+(\.[0-9]+)?$"
display(
    df_bronze_movies_metrics.filter(~F.col("popularity").rlike(padrao_numero))
      .select("popularity").distinct().limit(50)
)
display(
    df_bronze_movies_metrics.filter(~F.col("vote_average").rlike(padrao_numero))
      .select("vote_average").distinct().limit(50)
)